In [1]:
import pandas as pd

balls = pd.read_csv('/content/ball_by_ball_data.csv')
matches = pd.read_csv('/content/ipl_matches_data.csv')

Inspect Innings Structure

In [2]:
balls[['match_id','innings','over_number','ball_number']].head()

,match_id,innings,over_number,ball_number
0,335982,1,0,0
1,335982,1,0,1
2,335982,1,0,2
3,335982,1,0,3
4,335982,1,0,4


Create Match Outcome Table

In [3]:
match_results = matches[
['match_id','match_winner']
]

Create Over-Level Dataset

Currently:

1 row = 1 ball

Need:

1 row = 1 over

In [4]:
over_df = balls.groupby(
[
'match_id',
'innings',
'over_number',
'team_batting',
'team_bowling'
]
).agg(
runs=('total_runs','sum'),
wickets=('is_wicket','sum')
).reset_index()

Create Running Score

In [5]:
over_df['cumulative_runs'] = (
over_df.groupby(
['match_id','innings']
)['runs']
.cumsum()
)

Create Running Wickets

In [6]:
over_df['cumulative_wickets'] = (
over_df.groupby(
['match_id','innings']
)['wickets']
.cumsum()
)

Create Overs Completed

In [8]:
over_df['overs_completed'] = over_df['over_number'] + 1

Create Target Score

In [9]:
first_innings_total = (
balls[balls['innings']==1]
.groupby('match_id')
['total_runs']
.sum()
.reset_index()
)

Rename:

In [11]:
first_innings_total.columns = ['match_id','first_innings_score']

Merge:

In [13]:
over_df = over_df.merge(
first_innings_total,
on='match_id'
)

Create Chase Variables

Only for innings 2.

In [15]:
over_df['target'] = over_df['first_innings_score'] + 1

Runs required:

In [17]:
over_df['runs_required'] = over_df['target'] - over_df['cumulative_runs']

Balls remaining:

In [19]:
over_df['balls_remaining'] = (120 -(over_df['overs_completed']*6))

Create Run Rates

Current Run Rate:

In [20]:
over_df['crr'] = over_df['cumulative_runs'] / over_df['overs_completed']

Required Run Rate:

In [21]:
over_df['rrr'] =(over_df['runs_required'] /(over_df['balls_remaining']/6))

Create Momentum Features

Runs in last 3 overs

In [23]:
over_df['runs_last_3'] = (
    over_df.groupby(
        ['match_id', 'innings']
    )['runs']
    .rolling(3)
    .sum()
    .reset_index(level=[0, 1], drop=True)
)

Wickets in last 3 overs

In [25]:
over_df['wkts_last_3'] = (
over_df.groupby(
['match_id','innings']
)['wickets']
.rolling(3)
.sum()
.reset_index(level=[0, 1], drop=True)
)

Pressure:

In [26]:
over_df['pressure_index'] = over_df['rrr'] - over_df['crr']

Create Target Variable

Merge winner

In [27]:
over_df = over_df.merge(match_results, on='match_id')

Create binary target

In [28]:
over_df['win'] = (over_df['team_batting'] == over_df['match_winner']).astype(int)

from sklearn.linear_model import LogisticRegression

# Drop rows with NaN values that might have resulted from rolling calculations
# This is important for training a robust model
model_df = over_df.dropna(subset=features + ['win'])

X = model_df[features]
y = model_df['win']

model = LogisticRegression(solver='liblinear', random_state=42)
model.fit(X, y)

In [29]:
features = [
'cumulative_runs',
'cumulative_wickets',
'overs_completed',
'runs_required',
'balls_remaining',
'crr',
'rrr',
'pressure_index',
'runs_last_3',
'wkts_last_3'
]

Predict Probabilities

In [36]:
# Define features
features = [
    'cumulative_runs',
    'cumulative_wickets',
    'overs_completed',
    'runs_required',
    'balls_remaining',
    'crr',
    'rrr',
    'pressure_index',
    'runs_last_3',
    'wkts_last_3'
]

# Import Logistic Regression and train the model
from sklearn.linear_model import LogisticRegression
import numpy as np
import pandas as pd

# Replace infinite values with NaN before dropping them
# This is crucial because some calculations (like RRR, CRR) can produce inf when dividing by zero
over_df.replace([np.inf, -np.inf], np.nan, inplace=True)

# Drop rows with NaN values that might have resulted from rolling calculations or inf conversions
# This is important for training a robust model
model_df = over_df.dropna(subset=features + ['win'])

X = model_df[features]
y = model_df['win']

model = LogisticRegression(solver='liblinear', random_state=42)
model.fit(X, y)

# Initialize 'win_probability' column in over_df with NaN
over_df['win_probability'] = np.nan

# Identify rows in over_df that are clean (no NaNs in feature columns) for prediction
predictable_rows_mask = ~over_df[features].isna().any(axis=1)

# Get the clean subset of over_df for prediction
# Using .loc for explicit indexing to prevent SettingWithCopyWarning
over_df_clean_for_prediction = over_df.loc[predictable_rows_mask].copy()

# Predict probabilities for the clean subset
if not over_df_clean_for_prediction.empty:
    predictions = model.predict_proba(over_df_clean_for_prediction[features])[:, 1]
    # Assign predictions back to the 'win_probability' column in the original over_df
    over_df.loc[predictable_rows_mask, 'win_probability'] = predictions

Turning Point Engine

Calculate swing:

In [38]:
over_df['prob_shift'] = (
    over_df.groupby(
        'match_id'
    )['win_probability']
    .diff()
)

Flag major moments:

In [40]:
over_df['turning_point'] = (
    abs(
        over_df['prob_shift']
    ) > 0.20
)

In [41]:
over_df.to_csv(
'ipl_win_probability.csv',
index=False
)

In [42]:
over_df = over_df[over_df['innings'] == 2].copy()

In [43]:
over_df['target'] = over_df['first_innings_score'] + 1

over_df['runs_required'] = (
    over_df['target'] -
    over_df['cumulative_runs']
)

over_df['balls_remaining'] = (
    120 -
    (over_df['overs_completed'] * 6)
)

over_df['crr'] = (
    over_df['cumulative_runs'] /
    over_df['overs_completed']
)

over_df['rrr'] = (
    over_df['runs_required'] /
    (over_df['balls_remaining'] / 6)
)

over_df['pressure_index'] = (
    over_df['rrr'] -
    over_df['crr']
)

In [44]:
over_df[['match_id',
         'innings',
         'overs_completed',
         'cumulative_runs',
         'runs_required',
         'balls_remaining',
         'crr',
         'rrr',
         'pressure_index']].head(10)

,match_id,innings,overs_completed,cumulative_runs,runs_required,balls_remaining,crr,rrr,pressure_index
20,335982,2,1,4,219,114,4.000000,11.526316,7.526316
21,335982,2,2,9,214,108,4.500000,11.888889,7.388889
22,335982,2,3,12,211,102,4.000000,12.411765,8.411765
23,335982,2,4,16,207,96,4.000000,12.937500,8.937500
24,335982,2,5,24,199,90,4.800000,13.266667,8.466667
25,335982,2,6,26,197,84,4.333333,14.071429,9.738095
26,335982,2,7,33,190,78,4.714286,14.615385,9.901099
27,335982,2,8,38,185,72,4.750000,15.416667,10.666667
28,335982,2,9,43,180,66,4.777778,16.363636,11.585859
29,335982,2,10,51,172,60,5.100000,17.200000,12.100000


In [45]:
import numpy as np

over_df.replace([np.inf, -np.inf], np.nan, inplace=True)

over_df['rrr'] = over_df['rrr'].fillna(0)
over_df['pressure_index'] = over_df['pressure_index'].fillna(0)

In [46]:
over_df.to_csv('ipl_win_probability_updated.csv', index=False)

from google.colab import files
files.download('ipl_win_probability_updated.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>